# 02 — Walk-Forward Modeling

Expanding walk-forward evaluation with an **H-day embargo** (purges training rows
whose forward target overlaps the test block), and **per-fold** bucket edges fit on
training data only.

Both model families are reduced to a common **expected-return signal** so
classification and regression compete on the same footing:
- regressor -> predicted forward return is the signal;
- classifier -> `signal = sum_k P(k) * rbar_k` (per-fold bucket-mean returns).

Metrics: native ML metrics per type (balanced accuracy / log-loss; MAE / $R^2$) plus
the decision-relevant **Rank IC**, **non-overlapping Rank IC**, **ICIR**, and
**hit rate** on the signal. The stitched out-of-sample signals are cached for the
backtest (stage 03).

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt

from src.config import load_config
from src.experiment import run_experiment

cfg = load_config(PROJECT_ROOT / "config.yaml")

def _require(path, name):
    if not Path(path).exists():
        raise FileNotFoundError(f"{name} not found at {path}.\nRun 01_features_target.ipynb first.")
    return path

X = pd.read_parquet(_require(cfg.features_path, "features"))
targets = pd.read_parquet(_require(cfg.targets_path, "targets"))
warmup = max(cfg.return_windows)   # skip rows before long-window features exist
print(f"features: {X.shape} | targets: {list(targets.columns)} | warmup: {warmup}")

## 1 — Run (or load) the walk-forward

Results are cached to `oos_signals.parquet` + `walkforward_metrics.csv`. **Changing
`model:`, `target:`, or `features:` settings requires `force_refresh: true`** to rerun.

In [ ]:
if (not cfg.force_refresh) and cfg.signals_path.exists() and cfg.metrics_path.exists():
    metrics = pd.read_csv(cfg.metrics_path)
    signals = pd.read_parquet(cfg.signals_path)
    print("loaded cached walk-forward results")
else:
    metrics, signals = run_experiment(
        X, targets,
        horizons=cfg.horizons,
        model_names=cfg.model_names,
        n_folds=cfg.n_folds,
        min_train_frac=cfg.min_train_frac,
        n_buckets=cfg.n_buckets,
        bucket_method=cfg.bucket_method,
        fixed_bins=cfg.fixed_bins,
        warmup=warmup,
    )
    cfg.signals_path.parent.mkdir(parents=True, exist_ok=True)
    metrics.to_csv(cfg.metrics_path, index=False)
    signals.to_parquet(cfg.signals_path)
    print(f"saved -> {cfg.metrics_path.name}, {cfg.signals_path.name}")

## 2 — Metrics

In [ ]:
cols = ['horizon', 'model', 'kind', 'rank_ic', 'rank_ic_nonoverlap', 'icir',
        'hit_rate', 'balanced_accuracy', 'accuracy', 'log_loss', 'mae', 'rmse', 'r2']
cols = [c for c in cols if c in metrics.columns]
view = metrics[cols].copy()
num = view.select_dtypes('number').columns
view[num] = view[num].round(4)
view.sort_values(['horizon', 'rank_ic_nonoverlap'], ascending=[True, False])

## 3 — Rank IC by model and horizon

Non-overlapping Rank IC is the fairer read (it removes the overlap inflation). For a
daily-frequency single-index timing signal, small positive and *consistent* (high
ICIR) values are what matter, not large one-off correlations.

In [ ]:
hz = list(cfg.horizons.keys())
fig, axes = plt.subplots(1, len(hz), figsize=(5 * len(hz), 4), sharey=True)
if len(hz) == 1:
    axes = [axes]
for ax, h in zip(axes, hz):
    sub = metrics[metrics.horizon == h].set_index('model')['rank_ic_nonoverlap']
    colors = ['#3b7dd8' if v >= 0 else '#d1495b' for v in sub]
    sub.plot.bar(ax=ax, color=colors)
    ax.axhline(0, color='k', lw=0.8)
    ax.set_title(f'{h}  (H={cfg.horizons[h]})')
    ax.set_xlabel('')
    ax.set_ylabel('non-overlap Rank IC')
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

## 4 — Best model per horizon

Ranked by non-overlapping Rank IC. These signals are already saved; stage 03 will
turn a chosen one into positions, apply costs and roll, and produce the equity /
drawdown / regime visuals.

In [ ]:
best = (metrics.sort_values('rank_ic_nonoverlap', ascending=False)
               .groupby('horizon', as_index=False).first())
keep = [c for c in ['horizon', 'model', 'kind', 'rank_ic_nonoverlap', 'icir', 'hit_rate']
        if c in best.columns]
print(best[keep].to_string(index=False))
print(f"\nsignal columns saved for stage 03: "
      f"{[c for c in signals.columns if not c.startswith('realized__')]}")